## Construcción de stops
A partir de shapes de rutas se generan stops con distancia fija entre estaciones para cada ruta

In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

## Parámetros

In [2]:
import json
from pathlib import Path as PathLib

_params_path = PathLib.cwd() / "params.json"
if not _params_path.exists():
    _params_path = PathLib("params.json")
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

CIUDAD = p["ciudad"]
AGENCY_ID = p["agency"]["id"]
distancia_entre_estaciones = p["stops"]["distancia_entre_estaciones"]

In [3]:
# distancia_entre_estaciones cargada desde params.json en la celda anterior

rutas_entrada

In [4]:
# --- Carpeta GTFS ---
PATH_DIR_GTFS = Path(f"../data/{CIUDAD}/gtfs-output")
PATH_DIR_GTFS.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_GTFS.absolute()}")

# --- Carpeta proccesed ---
PATH_DIR_proccesed = Path(f"../data/{CIUDAD}/processed")
PATH_DIR_proccesed.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_proccesed.absolute()}")

Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../data/tampico/gtfs-output
Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../data/tampico/processed


## Lectura de archivos

### Leer shapes

In [5]:
routes_clean_path =  PATH_DIR_proccesed / "routes_clean.geojson"

routes_clean = gpd.read_file(routes_clean_path)
routes_clean.head()

,route_id,agency_id,route_short_name,route_long_name,route_type,shape_id,geometry
0,Route_1,IMEPLAN_Tampico,Route_1,Mirador - Aviación por Boulevard,3,Shape_Route_1,"LINESTRING (618098.532 2457141.016, 617928.009..."
1,Route_2,IMEPLAN_Tampico,Route_2,Tampico - Bosque Central de Abastos - Casa Blanca,3,Shape_Route_2,"LINESTRING (614915.384 2470530.991, 614995.778..."
2,Route_3,IMEPLAN_Tampico,Route_3,Madero - Revolución Verde - Central de Abastos,3,Shape_Route_3,"LINESTRING (614921.605 2470525.501, 615001.925..."
3,Route_9,IMEPLAN_Tampico,Route_9,Santa Elena - Colonias - Tampico,3,Shape_Route_9,"LINESTRING (618703.105 2469648.325, 618782.266..."
4,Route_4,IMEPLAN_Tampico,Route_4,Colosio - Águila Madero - Golfo,3,Shape_Route_4,"LINESTRING (613305.943 2469210.556, 613305.523..."


In [6]:
# Asegúrate de que está en 4326 (grados)
print(routes_clean.crs)  # debería decir EPSG:4326
# Reproyectar a UTM 14N (metros)
routes_m = routes_clean.to_crs(epsg=32614)

# Calcular longitud en metros
routes_m["length_m"] = routes_m.geometry.length

routes_m[["route_id", "route_short_name", "length_m"]].head()

EPSG:32614


,route_id,route_short_name,length_m
0,Route_1,Route_1,36485.117756
1,Route_2,Route_2,37079.950056
2,Route_3,Route_3,29924.023507
3,Route_9,Route_9,46477.036406
4,Route_4,Route_4,44973.208604


## Generar stops equidistantes para cada ruta y segmentos
Toma el linestring y lo divide en N segmentos

In [7]:
# Segmentos de longitud fija (metros). Stops en los extremos de cada segmento.
# Puedes usar distancia_entre_estaciones = 22 para paradas cada 22 m, o 200 para cada 200 m.
stops_rows = []
segments_rows = []

In [8]:
from shapely.ops import substring

def _line_substring(line, d0, d1):
    """Extrae el tramo exacto siguiendo las curvas de la línea."""
    return substring(line, d0, d1, normalized=False)

In [9]:
for idx, row in routes_m.iterrows():
    route_id = row["route_id"]
    route_short_name = row["route_short_name"]
    shape_id = f"shape_{route_short_name}"
    line = row["geometry"]
    if line is None or line.is_empty:
        continue
    total_m = line.length

    # #endregion
    n_segments = max(1, int(total_m // distancia_entre_estaciones))
    # Stops en 0, distancia_entre_estaciones, 2*distancia_entre_estaciones, ... y el último en min(n_segments * distancia_entre_estaciones, total_m)
    for stop_seq in range(n_segments + 1):
        measure_m = min(stop_seq * distancia_entre_estaciones, total_m)
        point = line.interpolate(measure_m)
        stop_id = f"{route_id}_{stop_seq:04d}"
        stops_rows.append({
            "route_id": route_id,
            "route_short_name": route_short_name,
            "shape_id": shape_id,
            "stop_seq": stop_seq,
            "measure_m": float(measure_m),
            "stop_id": stop_id,
            "geometry": point,
        })
    # Segmentos entre stops consecutivos
    for seg_seq in range(n_segments):
        from_m = seg_seq * distancia_entre_estaciones
        to_m = min((seg_seq + 1) * distancia_entre_estaciones, total_m)
        seg_geom = _line_substring(line, from_m, to_m)
        from_stop_id = f"{route_id}_{seg_seq:04d}"
        to_stop_id = f"{route_id}_{seg_seq + 1:04d}"
        segment_id = f"Seg_{route_id}_{seg_seq:04d}"
        length_m = to_m - from_m
        segments_rows.append({
            "route_id": route_id,
            "shape_id": shape_id,
            "segment_seq": seg_seq,
            "segment_id": segment_id,
            "from_stop_id": from_stop_id,
            "to_stop_id": to_stop_id,
            "from_measure_m": float(from_m),
            "to_measure_m": float(to_m),
            "length_m": float(length_m),
            "geometry": seg_geom,
        })

In [10]:

gdf_stops = gpd.GeoDataFrame(stops_rows, geometry="geometry", crs=routes_m.crs)
gdf_segments = gpd.GeoDataFrame(segments_rows, geometry="geometry", crs=routes_m.crs)

# Opcional: volver a WGS84 para uso en GTFS
gdf_stops = gdf_stops.to_crs(4326)
gdf_segments = gdf_segments.to_crs(4326)




# Formato tablas: columnas en el orden solicitado
gdf_stops = gdf_stops[["route_id", "route_short_name", "shape_id", "stop_seq", "measure_m", "stop_id", "geometry"]]
gdf_segments = gdf_segments[["route_id", "shape_id", "segment_seq", "segment_id", "from_stop_id", "to_stop_id",
                             "from_measure_m", "to_measure_m",  "geometry"]]




In [11]:
gdf_segments.head(3)

,route_id,shape_id,segment_seq,segment_id,from_stop_id,to_stop_id,from_measure_m,to_measure_m,geometry
0,Route_1,shape_Route_1,0,Seg_Route_1_0000,Route_1_0000,Route_1_0001,0.0,200.0,"LINESTRING (-97.85418 22.21563, -97.85504 22.2..."
1,Route_1,shape_Route_1,1,Seg_Route_1_0001,Route_1_0001,Route_1_0002,200.0,400.0,"LINESTRING (-97.85504 22.21401, -97.85586 22.2..."
2,Route_1,shape_Route_1,2,Seg_Route_1_0002,Route_1_0002,Route_1_0003,400.0,600.0,"LINESTRING (-97.85578 22.21244, -97.85486 22.2..."


In [12]:
gdf_stops.head(3)

,route_id,route_short_name,shape_id,stop_seq,measure_m,stop_id,geometry
0,Route_1,Route_1,shape_Route_1,0,0.0,Route_1_0000,POINT (-97.85418 22.21563)
1,Route_1,Route_1,shape_Route_1,1,200.0,Route_1_0001,POINT (-97.85504 22.21401)
2,Route_1,Route_1,shape_Route_1,2,400.0,Route_1_0002,POINT (-97.85578 22.21244)


## Format stops to GTFS format

In [13]:
stops_gtfs = gdf_stops.copy()

In [14]:
stops_gtfs["stop_name"] = stops_gtfs["stop_id"]

stops_gtfs["stop_lon"] = stops_gtfs["geometry"].x
stops_gtfs["stop_lat"] = stops_gtfs["geometry"].y

#stops.drop(columns=["route_id",  "shape_id",   "geometry"], inplace=True)
stops_gtfs = stops_gtfs[['stop_id', 'stop_name', 'stop_lon', 'stop_lat']]
stops_gtfs.head()

,stop_id,stop_name,stop_lon,stop_lat
0,Route_1_0000,Route_1_0000,-97.854180,22.215630
1,Route_1_0001,Route_1_0001,-97.855039,22.214010
2,Route_1_0002,Route_1_0002,-97.855778,22.212444
3,Route_1_0003,Route_1_0003,-97.854412,22.213094
4,Route_1_0004,Route_1_0004,-97.853544,22.214710


In [15]:
stops_gtfs.tail()

,stop_id,stop_name,stop_lon,stop_lat
23277,Route_120_0153,Route_120_0153,-97.910517,22.395765
23278,Route_120_0154,Route_120_0154,-97.910248,22.397554
23279,Route_120_0155,Route_120_0155,-97.909916,22.399300
23280,Route_120_0156,Route_120_0156,-97.909591,22.400503
23281,Route_120_0157,Route_120_0157,-97.910016,22.402266


## Export

In [16]:
# archivos procesamiento
gdf_segments.to_file(PATH_DIR_proccesed / "segments.geojson", driver="GeoJSON")
gdf_stops.to_file(PATH_DIR_proccesed / "stops.geojson", driver="GeoJSON")


In [17]:
stops_gtfs.to_csv(PATH_DIR_GTFS / "stops.txt", index=False)